In [1]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D

import os
import pandas as pd
import numpy as np

In [2]:
train_dir = 'dataset/train'
test_dir = 'dataset/test'

In [3]:
def createdataframe(dir):
    image_paths=[]
    labels=[]
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir, label)):
            image_paths.append(os.path.join(dir, label, imagename))
            labels.append(label)

        print(label, "completed")

    return image_paths, labels

In [4]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(train_dir)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
print(train)

                                              image     label
0         dataset/train\angry\Training_10118481.jpg     angry
1         dataset/train\angry\Training_10120469.jpg     angry
2         dataset/train\angry\Training_10131352.jpg     angry
3         dataset/train\angry\Training_10161559.jpg     angry
4          dataset/train\angry\Training_1021836.jpg     angry
...                                             ...       ...
28704  dataset/train\surprise\Training_99916297.jpg  surprise
28705  dataset/train\surprise\Training_99924420.jpg  surprise
28706  dataset/train\surprise\Training_99937001.jpg  surprise
28707  dataset/train\surprise\Training_99951755.jpg  surprise
28708  dataset/train\surprise\Training_99984132.jpg  surprise

[28709 rows x 2 columns]


In [6]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(test_dir)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [7]:
print(test)

                                              image     label
0       dataset/test\angry\PrivateTest_10131363.jpg     angry
1       dataset/test\angry\PrivateTest_10304478.jpg     angry
2        dataset/test\angry\PrivateTest_1054527.jpg     angry
3       dataset/test\angry\PrivateTest_10590091.jpg     angry
4        dataset/test\angry\PrivateTest_1109992.jpg     angry
...                                             ...       ...
7173  dataset/test\surprise\PublicTest_98089595.jpg  surprise
7174  dataset/test\surprise\PublicTest_98567249.jpg  surprise
7175  dataset/test\surprise\PublicTest_98972870.jpg  surprise
7176  dataset/test\surprise\PublicTest_99242645.jpg  surprise
7177  dataset/test\surprise\PublicTest_99446963.jpg  surprise

[7178 rows x 2 columns]


In [8]:
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import load_img
import numpy as np

def extract_features(images):
    features = []

    for image in tqdm(images):
        img = load_img(image, color_mode='grayscale', target_size=(48, 48))
        img = np.array(img)
        features.append(img)

    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)

    return features

In [9]:
train_features = extract_features(train['image'])

100%|██████████| 28709/28709 [01:32<00:00, 311.92it/s]


In [10]:
test_features = extract_features(test['image'])

100%|██████████| 7178/7178 [00:21<00:00, 329.25it/s]


In [11]:
x_train = train_features / 255.0
x_test = test_features / 255.0

In [12]:
from sklearn.preprocessing import LabelEncoder

In [13]:
le = LabelEncoder()

y_train = le.fit_transform(train['label'])
y_test = le.transform(test['label'])

y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [16]:
model = Sequential()

# Convolutional layers
model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

model.add(Flatten())

# Fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# Output layer
model.add(Dense(7, activation='softmax'))

In [18]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=20,
    validation_data=(x_test, y_test)
)

Epoch 1/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 442s 2s/step - accuracy: 0.4993 - loss: 1.3118 - val_accuracy: 0.5220 - val_loss: 1.2275
Epoch 2/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 372s 2s/step - accuracy: 0.5080 - loss: 1.2927 - val_accuracy: 0.5504 - val_loss: 1.1824
Epoch 3/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 366s 2s/step - accuracy: 0.5148 - loss: 1.2770 - val_accuracy: 0.5518 - val_loss: 1.1769
Epoch 4/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 366s 2s/step - accuracy: 0.5211 - loss: 1.2590 - val_accuracy: 0.5552 - val_loss: 1.1628
Epoch 5/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 360s 2s/step - accuracy: 0.5300 - loss: 1.2449 - val_accuracy: 0.5578 - val_loss: 1.1607
Epoch 6/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 2186s 10s/step - accuracy: 0.5296 - loss: 1.2324 - val_accuracy: 0.5592 - val_loss: 1.1520
Epoch 7/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 340s 2s/step - accuracy: 0.5409 - loss: 1.2141 - val_accuracy: 0.5751 - val_loss: 1.1337
Epoch 8/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 339s 2s/step - accuracy: 0.5428 - loss: 1.2032 - val_ac

In [19]:
model_json = model.to_json()
with open("emotiondetector.json",'w') as json_file:
    json_file.write(model_json)
    model.save("emotiondetector.h5")

In [20]:
from keras.models import model_from_json

In [22]:
json_file = open("emotiondetector.json", "r")
model_json = json_file.read()
json_file.close()
model = model_from_json(model_json)
model.load_weights("emotiondetector.h5")

In [23]:
label = ['angry','disgust','fear','happy','neutral','sad','surprise']

In [26]:
def ef(image):
    img = load_img(image, color_mode="grayscale", target_size=(48, 48) )
    feature = np.array(img)
    feature = feature.reshape(1,48,48,1)
    return feature/255.0

In [35]:
image = r"C:\Users\user\Downloads\face_emotion_detection\dataset\train\sad\Training_91012494.jpg"
print("original image is of sad")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)

original image is of sad
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
model prediction is  sad


In [33]:
import matplotlib.pyplot as plt
%matplotlib inline

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
image = 'dataset/train/sad/42.jpg'
print("original image is of sad")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)
plt.imshow(img.reshape(48,48),cmap='gray')

In [ ]:
image = 'dataset/train/fear/2.jpg'
print("original image is of fear")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)
plt.imshow(img.reshape(48,48),cmap='gray')